In [1]:
import sys
from pathlib import Path

class ImportMyModules:

    def __init__(self) -> None:
        self.cwd = Path.cwd().resolve()

    def _get_repo_root(self, folder: str) -> Path:
        return next(
            parent for parent in [self.cwd, *self.cwd.parents]
            if (parent / "lib" / "peregrin" / folder).exists()
        )

    def insert_path(self, folder: str) -> None:
        repo_root = self._get_repo_root("src")

        package_root = repo_root / "lib" / "peregrin"
        print(f"Adding {package_root} to sys.path")
        sys.path.insert(0, str(package_root))

importer = ImportMyModules()
importer.insert_path("src")
importer.insert_path("data")

Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path
Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path


In [2]:
path_xml = r"C:\Users\modri\Desktop\img_sq.xml"

In [3]:
path_csv = r"C:\Users\modri\Desktop\position_0003440_allspots.csv"

In [4]:
import numpy as np
import pandas as pd


def craft_dummy_dataframe(
    n_tracks: int = 6,
    n_time_points: int = 100,
    seed: int | None = 20,
    save: bool = False,
) -> pd.DataFrame:
    """
    Generate a synthetic dataset simulating cell trajectories,
    organised into 2 sets, each with 2 subsets.

    Cells perform a persistent random walk (correlated random walk) with
    per-cell speed and directional persistence, plus small positional noise,
    which mimics realistic migrating-cell behaviour.

    Returns a DataFrame with columns:
        'set', 'subset', 'track_id', 'time_point',
        'x_coordinate', 'y_coordinate'
    """
    rng = np.random.default_rng(seed)
    records = []

    sets = ['set_A', 'set_B']
    subsets = ['subset_1', 'subset_2']

    global_track_id = 0

    for set_name in sets:
        for subset_name in subsets:
            for _ in range(n_tracks):
                global_track_id += 1
                track_id = global_track_id

                # Each cell can appear/disappear at slightly different times
                start = rng.integers(0, n_time_points // 4)
                length = rng.integers(n_time_points // 2, n_time_points + 1)
                end = min(start + length, n_time_points)

                # Random starting position in a 2D field
                x, y = rng.uniform(0, 500), rng.uniform(0, 500)

                # Per-cell motility characteristics
                speed = rng.uniform(2.0, 8.0)          # mean step length
                persistence = rng.uniform(0.6, 0.98)   # directional memory (0-1)
                angle = rng.uniform(0, 2 * np.pi)      # initial heading

                for t in range(start, end):
                    records.append(
                        (set_name, subset_name, track_id, t, x, y)
                    )

                    # Update heading with persistence (correlated random walk)
                    angle += (1 - persistence) * rng.uniform(-np.pi, np.pi)

                    step = rng.normal(speed, speed * 0.2)
                    x += step * np.cos(angle) + rng.normal(0, 0.5)
                    y += step * np.sin(angle) + rng.normal(0, 0.5)

    df = pd.DataFrame(
        records,
        columns=[
            'set', 'subset', 'track_id',
            'time_point', 'x_coordinate', 'y_coordinate',
        ],
    )
    df = df.sort_values(
        ['set', 'subset', 'track_id', 'time_point']
    ).reset_index(drop=True)

    df['track_uid'] = df.groupby(['set', 'subset', 'track_id']).ngroup()

    if save:
        df.to_csv("dummy_cell_tracks.csv", index=False)

    return df



df = craft_dummy_dataframe(save=True)
path_csv_naked = r".\dummy_cell_tracks.csv"
df

,set,subset,track_id,time_point,x_coordinate,y_coordinate,track_uid
0,set_A,subset_1,1,22,230.573355,60.859846,0
1,set_A,subset_1,1,23,234.227119,60.628657,0
2,set_A,subset_1,1,24,239.168127,60.344644,0
3,set_A,subset_1,1,25,242.898700,57.809882,0
4,set_A,subset_1,1,26,245.228029,54.597547,0
...,...,...,...,...,...,...,...
1708,set_B,subset_2,24,95,165.663738,651.145494,23
1709,set_B,subset_2,24,96,160.947074,653.762386,23
1710,set_B,subset_2,24,97,151.214694,656.336399,23
1711,set_B,subset_2,24,98,146.936610,659.480518,23


In [ ]:
import src.loader.load as load_module
from importlib import reload

reload(load_module)
load_data = load_module.load_data

In [6]:
data_xml = load_data(path_xml)

In [7]:
data_csv = load_data(path_csv)

In [8]:
data_csv_converted = load_data(
    path_csv,
    convert_spatial_to='nm', convert_time_to='min'
)

In [9]:
data_csv_naked = load_data(
    path_csv_naked, 
    {
        'id': 'track_id',
        't': 'time_point',
        'x': 'x_coordinate',
        'y': 'y_coordinate',
    },
)

C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\loader\load.py:102: InputWarning: No time units found in input files.
 Please specify the time units using <load_data result>.metadata.write(timeunits="<unit>")
  self._check()
C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\loader\load.py:102: InputWarning: No spatial units found in input files.
 Please specify the spatial units using <load_data result>.metadata.write(spatialunits="<unit>")
  self._check()


In [10]:
data_csv_naked.metadata.write(spatialunits='microns', timeunits='seconds')

In [11]:
data_csv_custom = load_data(
    path_csv_naked, 
    {
        'id': 'track_id',
        't': 'time_point',
        'x': 'x_coordinate',
        'y': 'y_coordinate',
    },
    spatial_unit='meter', time_unit='sec'
)

In [12]:
data = [data_xml, data_csv, data_csv_converted, data_csv_naked, data_csv_custom]

In [13]:
for d in data:
    print(d.metadata.get())
    print(d.metadata.get_each())

{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': '90.0', 'nframes': '90'}
{'img_sq.xml': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': '90.0', 'nframes': '90'}}
{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90}
{'position_0003440_allspots.csv': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90}}
{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90}
{'position_0003440_allspots.csv': {'TRACK_ID': '', 'POSITION_T': 's', 'POSITION_X': 'μm', 'POSITION_Y': 'μm', 'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 90.0, 'nframes': 90}, 'spatialunits': 'nm', 'timeinterval': 1.5, 'timeunits': 'min'}
{'spatialunits': 'μm', 'timeunits': 's', 'timeinterval': 1.0, 'nframes': 99}
{'dummy_cell_tracks.csv': {'track_id': '', 'time_point': '', 'x_coordinat